# Feature Engineering

In [1]:
import pandas as pd 
import numpy as np 

df = pd.read_csv("emi_prediction_cleaned.csv")
print("Shape:", df.shape)
df.head() 

Shape: (388233, 27)


,age,gender,marital_status,education,monthly_salary,employment_type,years_of_employment,company_type,house_type,monthly_rent,...,existing_loans,current_emi_amount,credit_score,bank_balance,emergency_fund,emi_scenario,requested_amount,requested_tenure,emi_eligibility,max_monthly_emi
0,38.0,Female,Married,Professional,82600.0,Private,0.9,Mid-size,Rented,20000.0,...,Yes,23700.0,660.0,303200.0,70200.0,Personal Loan EMI,850000.0,15,Not_Eligible,500.0
1,38.0,Female,Married,Graduate,21500.0,Private,7.0,MNC,Family,0.0,...,Yes,4100.0,714.0,92500.0,26900.0,E-commerce Shopping EMI,128000.0,19,Not_Eligible,700.0
2,38.0,Male,Married,Professional,86100.0,Private,5.8,Startup,Own,0.0,...,No,0.0,650.0,672100.0,324200.0,Education EMI,306000.0,16,Eligible,27775.0
3,58.0,Female,Married,High School,66800.0,Private,2.2,Mid-size,Own,0.0,...,No,0.0,685.0,440900.0,178100.0,Vehicle EMI,304000.0,83,Eligible,16170.0
4,48.0,Female,Married,Professional,57300.0,Private,3.4,Mid-size,Family,0.0,...,No,0.0,770.0,97300.0,28200.0,Home Appliances EMI,252000.0,7,Not_Eligible,500.0


In [2]:
# Check the target column
print(df["emi_eligibility"].value_counts())

emi_eligibility
Not_Eligible    300086
Eligible         71380
High_Risk        16767
Name: count, dtype: int64


In [20]:
# Separate X and y
# Separate features and target

X = df.drop("emi_eligibility", axis=1)
y = df["emi_eligibility"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (388233, 29)
y shape: (388233,)

Target distribution:
emi_eligibility
Not_Eligible    300086
Eligible         71380
High_Risk        16767
Name: count, dtype: int64


# Here:

X = input features

y = what the model needs to predict

In [4]:
# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:")
print(numerical_cols)

print("\nCategorical columns:")
print(categorical_cols)

Numerical columns:
['age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'current_emi_amount', 'credit_score', 'emergency_fund', 'requested_amount', 'requested_tenure', 'max_monthly_emi']

Categorical columns:
['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'existing_loans', 'bank_balance', 'emi_scenario']


In [5]:
# Encode categorical features
X_encoded = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

print("Shape after encoding:", X_encoded.shape)

Shape after encoding: (388233, 12190)


In [6]:
X_encoded.head()

,age,monthly_salary,years_of_employment,monthly_rent,family_size,dependents,school_fees,college_fees,travel_expenses,groceries_utilities,...,bank_balance_999300.0,bank_balance_999400.0,bank_balance_999600.0,bank_balance_999700.0,bank_balance_999800.0,bank_balance_nan.0,emi_scenario_Education EMI,emi_scenario_Home Appliances EMI,emi_scenario_Personal Loan EMI,emi_scenario_Vehicle EMI
0,38.0,82600.0,0.9,20000.0,3,2,0.0,0.0,7200.0,19500.0,...,False,False,False,False,False,False,False,False,True,False
1,38.0,21500.0,7.0,0.0,2,1,5100.0,0.0,1400.0,5400.0,...,False,False,False,False,False,False,False,False,False,False
2,38.0,86100.0,5.8,0.0,4,3,0.0,0.0,10200.0,19400.0,...,False,False,False,False,False,False,True,False,False,False
3,58.0,66800.0,2.2,0.0,5,4,11400.0,0.0,6200.0,11900.0,...,False,False,False,False,False,False,False,False,False,True
4,48.0,57300.0,3.4,0.0,4,3,9400.0,21300.0,3600.0,16200.0,...,False,False,False,False,False,False,False,True,False,False


In [8]:
print(y.unique())

[nan  1.]


In [7]:
# Encode the target
y = y.map({
    "Eligible": 1,
    "Not Eligible": 0
})

In [9]:
print(y.value_counts())

emi_eligibility
1.0    71380
Name: count, dtype: int64


# Feature creation

Total monthly expenses

In [10]:
df["total_monthly_expenses"] = (
    df["monthly_rent"] +
    df["school_fees"] +
    df["college_fees"] +
    df["travel_expenses"]
)

Disposable income

In [11]:
df["disposable_income"] = (
    df["monthly_salary"] -
    df["total_monthly_expenses"]
)

Expense-to-income ratio

In [12]:
df["expense_to_income_ratio"] = (
    df["total_monthly_expenses"] /
    df["monthly_salary"]
)

Check the engineered dataset

In [13]:
print("Final shape:", X_encoded.shape)

print("\nMissing values:")
print(X_encoded.isnull().sum().sum())

print("\nData types:")
print(X_encoded.dtypes.value_counts())


Final shape: (388233, 12190)

Missing values:
0

Data types:
bool       12173
float64       14
int64          3
Name: count, dtype: int64


Save the engineered data

In [15]:
print("X exists:", "X" in globals())
print("y exists:", "y" in globals())

X exists: True
y exists: True


In [21]:
from sklearn.preprocessing import LabelEncoder

X_encoded = X.copy()

categorical_cols = X_encoded.select_dtypes(include=["object"]).columns

print("Categorical columns:")
print(list(categorical_cols))

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))

print("\nEncoding completed!")
print("X_encoded shape:", X_encoded.shape)

Categorical columns:
['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'existing_loans', 'bank_balance', 'emi_scenario']

Encoding completed!
X_encoded shape: (388233, 29)


In [22]:
# Encode the target separately
target_encoder = LabelEncoder()

y_encoded = target_encoder.fit_transform(y)

print("Target classes:")
print(target_encoder.classes_)

print("\nEncoded target distribution:")
print(pd.Series(y_encoded).value_counts())

Target classes:
['Eligible' 'High_Risk' 'Not_Eligible']

Encoded target distribution:
2    300086
0     71380
1     16767
Name: count, dtype: int64


In [23]:
final_df = X_encoded.copy()
final_df["emi_eligibility"] = y_encoded

print("Final dataset shape:", final_df.shape)
print("\nMissing target values:",
      final_df["emi_eligibility"].isna().sum())

Final dataset shape: (388233, 30)

Missing target values: 0


In [24]:
# Save it
final_df.to_csv(
    "feature_engineered_dataset.csv",
    index=False
)

print("Feature engineered dataset saved successfully!")

Feature engineered dataset saved successfully!
